# Spin-polarized spectra

The same three systems as `easy_check_polarization.m`: ISC triplet, recombination triplet and singlet-born radical pair. Run the MATLAB script for EasySpin overlays. CSV columns are field (T), intensity.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import mars

dtype = torch.float64
fields = torch.linspace(0.3, 0.4, 1024, dtype=dtype)
MARS_SPECTRA = Path("mars_spectra")
EASYSPIN_SPECTRA = Path("easyspin_spectra")
MARS_SPECTRA.mkdir(exist_ok=True)
EASYSPIN_SPECTRA.mkdir(exist_ok=True)


In [ ]:
def plot_spectrum(name, sample, context):
    creator = mars.spectra_manager.StationarySpectra(
        sample=sample, context=context, freq=9.8e9,
        harmonic=0, dtype=dtype,
    )
    spectrum = creator(sample, fields=fields)
    np.savetxt(MARS_SPECTRA / f"{name}.csv",
               np.column_stack((fields.numpy(), spectrum.detach().numpy())), delimiter=",")
    plt.figure()
    plt.plot(fields, spectrum / spectrum.abs().max(), label="MaRs")
    reference = EASYSPIN_SPECTRA / f"{name}.csv"
    if reference.exists():
        field_easyspin, spectrum_easyspin = np.loadtxt(reference, delimiter=",").T
        plt.plot(field_easyspin, spectrum_easyspin / np.abs(spectrum_easyspin).max(),
                 "--", label="EasySpin")
    plt.xlabel("Field (T)")
    plt.ylabel("Normalized intensity")
    plt.title(name)
    plt.legend()
    plt.show()


def triplet_sample(ham_strain=0.0):
    spin_system = mars.spin_model.SpinSystem(
        electrons=[1],
        g_tensors=[mars.spin_model.Interaction([2.02, 2.02, 2.02], dtype=dtype)],
        electron_electron=[(0, 0, mars.spin_model.DEInteraction([500e6, 100e6], dtype=dtype))],
        dtype=dtype,
    )
    return mars.spin_model.SolidSample(
        base_spin_system=spin_system, gauss=1e-3, lorentz=1e-3,
        ham_strain=ham_strain, mesh=(20, 40), dtype=dtype,
    )


### 1. ISC triplet

In [ ]:
sample = triplet_sample(ham_strain=1e6)

# Columns are Tx, Ty, Tz in the product basis |+1>, |0>, |-1>.
xyz = torch.tensor([[-1/np.sqrt(2), 1j/np.sqrt(2), 0],
                    [0, 0, 1],
                    [1/np.sqrt(2), 1j/np.sqrt(2), 0]], dtype=torch.complex128)
density = xyz @ torch.diag(torch.tensor([0.2, 0.3, 0.5], dtype=torch.complex128)) @ xyz.conj().T
context = mars.population.Context(sample=sample, basis="product", init_density=density, dtype=dtype)
plot_spectrum("isc_triplet", sample, context)


### 2. Recombination triplet

In [ ]:
sample = triplet_sample()
context = mars.population.Context(
    sample=sample, basis="eigen", init_populations=[0, 1, 0], dtype=dtype,
)
plot_spectrum("recombination_triplet", sample, context)


### 3. Spin-correlated radical pair

In [ ]:
spin_system = mars.spin_model.SpinSystem(
    electrons=[0.5, 0.5],
    g_tensors=[mars.spin_model.Interaction([2.02, 2.03, 2.04], dtype=dtype),
               mars.spin_model.Interaction([2.03, 2.025, 2.025], dtype=dtype)],
    electron_electron=[(0, 1, mars.spin_model.Interaction(
        [400e6 - 23.3333333333e6, 400e6 - 43.3333333333e6, 400e6 + 66.6666666667e6], dtype=dtype))],
    dtype=dtype,
)
sample = mars.spin_model.SolidSample(
    base_spin_system=spin_system, gauss=1e-3, lorentz=1e-3,
    mesh=(20, 40), dtype=dtype,
)
singlet = torch.tensor([0, 1, -1, 0], dtype=torch.complex128) / np.sqrt(2)
density = torch.outer(singlet, singlet.conj())
context = mars.population.Context(sample=sample, basis="product", init_density=density, dtype=dtype)
plot_spectrum("radical_pair", sample, context)
